In [1]:
library(fixest)
library(dplyr)
library(tidyr)
library(ggplot2)
library(lubridate)
library(modelsummary)

panel <- read.csv("../data/monthly_panel_mttu_mttr.csv")
raw_df <- read.csv("../data/sc_control_mttu_mttr_data_final_with_agg_score_and_vuln_count.csv")

raw_df <- raw_df %>%
    mutate(year_month = floor_date(as_date(published_at), "month"))


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘lubridate’


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union




## MTTU

In [ ]:
# Dichotomize variables 

bin_zero_pos <- function(x) {              # adoption: 0 vs any positive
  factor(if_else(x < 0.5, "zero", "positive"),
         levels = c("zero", "positive"))
}

bin_ten_less <- function(x) {              # regression: full marks vs not
  factor(if_else(x > 9.5, "ten", "below_ten"),
         levels = c("below_ten", "ten"))
}


binnings <- list(zero_pos = bin_zero_pos, ten_less = bin_ten_less)

chosen_binning <- c(
  # Aggregate_score            = "continuous",
  Maintained_score           = "continuous",
  Code_Review_score          = "continuous",
  Pinned_Dependencies_score  = "zero_pos",
  Security_Policy_score      = "zero_pos",
  Token_Permissions_score    = "zero_pos",
  DependUpTool_score         = "zero_pos",
  Binary_Artifacts_score     = "ten_less",
  Dangerous_Workflow_score   = "ten_less",
  Contrib_score              = "ten_less"
)

binned_metrics <- chosen_binning[chosen_binning != "continuous"]
continuous_metrics <- names(chosen_binning[chosen_binning == "continuous"])

# 1. Create binned columns in panel
panel_binned <- panel
for (m in names(binned_metrics)) {
  f <- binnings[[binned_metrics[[m]]]]
  panel_binned[[paste0(m, "_bin")]] <- f(panel_binned[[m]])
}

# 2. Build formula: binned versions for the binned metrics,
# raw continuous columns for Maintained_score and Code_Review_score
# Aggregate_score is excluded from this model 
bin_terms  <- paste(paste0(names(binned_metrics), "_bin"), collapse = " + ")
cont_terms <- paste(continuous_metrics, collapse = " + ")

fml_all_binned_str <- paste(
  "MTTU ~",
  paste(c(cont_terms, bin_terms), collapse = " + "), "+",
  controls,
  "|", fe_part
)
fml_all_binned <- as.formula(fml_all_binned_str)

# 3. Run the model
model_all_binned <- feglm(fml_all_binned, data = panel_binned,
                           family = "poisson", cluster = ~package_name)
summary(model_all_binned)

NOTES: 11,248 observations removed because of NA values (RHS: 11,248).
       4,414/0 fixed-effects (13,575 observations) removed because of only 0 outcomes or singletons.



GLM estimation, family = poisson, Dep. Var.: MTTU
Observations: 54,240
Fixed-effects: package_name: 7,224,  year_month: 25
Standard-errors: Clustered (package_name) 
                                       Estimate Std. Error    z value
Maintained_score                      -0.037373   0.003355 -11.138211
Code_Review_score                      0.005195   0.004080   1.273322
Pinned_Dependencies_score_binpositive -0.041283   0.048342  -0.853981
Security_Policy_score_binpositive     -0.136418   0.059024  -2.311240
Token_Permissions_score_binpositive    0.007998   0.051485   0.155338
DependUpTool_score_binpositive        -0.021063   0.049296  -0.427277
Binary_Artifacts_score_binten         -0.082960   0.072057  -1.151313
Dangerous_Workflow_score_binten        0.015621   0.053782   0.290460
Contrib_score_binten                  -0.002755   0.083584  -0.032965
log1p(downloads_7_day_total)          -0.006595   0.003671  -1.796339
log1p(dependents)                      0.019471   0.010243   1.9